In [ ]:
# The goal of this notebook is to analyse stats over the season,, majorly using table 
#Player_gameweek_score

In [20]:
import pymysql
import pandas as pd
import os
from sqlalchemy import Integer, String, create_engine, select,text,distinct
from sqlalchemy.orm import Session,DeclarativeBase,sessionmaker
from sqlalchemy import URL

a = pymysql.connect(host="localhost",database="fpl", user="root", password="password")

In [21]:
def create_connection_engine(db, host = "localhost", user = "root", password = "password"):
    """Creates a SQLAlchemy engine with a mysql database"""

    url_object = URL.create(
        "mysql+pymysql",
        username=user,
        password=password, 
        host=host,
        database=db)

    return create_engine(url_object)

session = sessionmaker(create_connection_engine('fpl'))

READING FROM DATABASE

In [32]:
query_1 = text("select * from Player_gameweek_score")
obj = session.execute(query_1).all()

columns = ["index","player_id","minutes","goals_scored","assists","clean_sheets","goals_conceded","own_goals","penalties_saved","penalties_missed","yellow_cards","red_cards","saves","bonus","bps ","influence","creativity","threat","ict_index","starts","expected_goals","expected_assists","expected_goal_involvements","expected_goals_conceded","total_points","in_dreamteam","gameweek","team","position"]

df = pd.DataFrame(obj, columns=columns)
df = df.drop(labels="index", axis = 1)
df = df.set_index("player_id")

print(len(df))
df.head()

28804


,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,penalties_missed,yellow_cards,red_cards,...,starts,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,total_points,in_dreamteam,gameweek,team,position
player_id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0.00,0.00,0.00,0.00,0,0,1,Arsenal,Forward
2,0,0,0,0,0,0,0,0,0,0,...,0,0.00,0.00,0.00,0.00,0,0,1,Arsenal,Defender
3,0,0,0,0,0,0,0,0,0,0,...,0,0.00,0.00,0.00,0.00,0,0,1,Arsenal,Midfielder
4,0,0,0,0,0,0,0,0,0,0,...,0,0.00,0.00,0.00,0.00,0,0,1,Arsenal,Midfielder
5,4,0,0,0,0,0,0,0,0,0,...,0,0.00,0.00,0.00,0.02,1,0,1,Arsenal,Defender


In [92]:
#fixture_df

fixture_query = text("select * from Fixtures_table")
obj = session.execute(fixture_query).all()

columns = ["index","code","event","finished","fixture_id","kickoff_time","minutes","team_a","team_a_score","team_h_score","team_h","team_h_difficulty","team_a_difficulty"]
fixtures_df = pd.DataFrame(obj, columns= columns)


fixtures_df.head()

,index,code,event,finished,fixture_id,kickoff_time,minutes,team_a,team_a_score,team_h_score,team_h,team_h_difficulty,team_a_difficulty,month,year
0,0,2367538,1,1,1,2023-08-11 19:00:00,90,13,3.0,0.0,6,5,2,August,2023
1,1,2367540,1,1,2,2023-08-12 12:00:00,90,16,1.0,2.0,1,2,5,August,2023
2,2,2367539,1,1,3,2023-08-12 14:00:00,90,19,1.0,1.0,3,2,2,August,2023
3,3,2367541,1,1,4,2023-08-12 14:00:00,90,12,1.0,4.0,5,2,3,August,2023
4,4,2367542,1,1,5,2023-08-12 14:00:00,90,10,1.0,0.0,9,2,2,August,2023


In [65]:
#Player to team relation for 1st half and 2nd half of the season

team_1_query = text("select * from EPL_PLAYERS_2023_1ST_HALF")
team_2_query = text("select * from EPL_PLAYERS_2023_2ND_HALF")

obj = session.execute(team_1_query).all()

columns_1 = ["player_id","team","position","player_name","team_code"]
columns_2 = ["player_id","team_code","team","position","player_name"]
team_1_df = pd.DataFrame(obj, columns= columns_1)

obj = session.execute(team_2_query).all()
team_2_df = pd.DataFrame(obj, columns= columns_2)

team_1_df = team_1_df[columns_2]

,player_id,team_code,team,position,player_name
0,1,1,Arsenal,Forward,Folarin Balogun
1,2,1,Arsenal,Defender,Cédric Alves Soares
2,3,1,Arsenal,Midfielder,Mohamed Elneny
3,4,1,Arsenal,Midfielder,Fábio Ferreira Vieira
4,5,1,Arsenal,Defender,Gabriel dos Santos Magalhães


COMBINING TABLES

In [62]:
team_code_df = team_2_df[['team', 'team_code']].drop_duplicates()
team_code_key = {i.team_code:i.team for i in team_code_df.itertuples()}

del team_code_df
team_code_key

try :
    fixtures_df['team_a'] = fixtures_df['team_a'].map(lambda x: team_code_key[x])
except KeyError:
    pass

try :
    fixtures_df['team_h'] = fixtures_df['team_h'].map(lambda x: team_code_key[x])
except KeyError:
    pass

del team_code_key

fixtures_df["kickoff_time"] = fixtures_df["kickoff_time"].astype("datetime64[h]")


MONTHS = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
fixtures_df['month'] = fixtures_df['kickoff_time'].map(lambda x: MONTHS[x.month - 1])
fixtures_df['year'] = fixtures_df['kickoff_time'].map(lambda x: x.year)

fixtures_df.head()

,index,code,event,finished,fixture_id,kickoff_time,minutes,team_a,team_a_score,team_h_score,team_h,team_h_difficulty,team_a_difficulty
0,0,2367538,1,1,1,2023-08-11T19:00:00Z,90,Man City,3.0,0.0,Burnley,5,2
1,1,2367540,1,1,2,2023-08-12T12:00:00Z,90,Nott'm Forest,1.0,2.0,Arsenal,2,5
2,2,2367539,1,1,3,2023-08-12T14:00:00Z,90,West Ham,1.0,1.0,Bournemouth,2,2
3,3,2367541,1,1,4,2023-08-12T14:00:00Z,90,Luton,1.0,4.0,Brighton,2,3
4,4,2367542,1,1,5,2023-08-12T14:00:00Z,90,Fulham,1.0,0.0,Everton,2,2


In [ ]:
player_id_name_df = team_1_df[['player_id', 'player_name']].drop_duplicates()
player_id_name_2_df = team_2_df[['player_id', 'player_name']].drop_duplicates()

player_id_key = {i.player_id:i.player_name for i in player_id_name_df.itertuples()}
player_id_key_2 = {i.player_id:i.player_name for i in player_id_name_2_df.itertuples()}

del player_id_name_df
del player_id_name_2_df

#player_id_key
try:
    df['player_name'] = df.index.map(lambda x: player_id_key[x])
except KeyError:
    df['player_name'] = df.index.map(lambda x: player_id_key_2[x])

del team_1_df
del team_2_df


In [97]:
df.head()

,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,penalties_missed,yellow_cards,red_cards,...,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,total_points,in_dreamteam,gameweek,team,position,player_name
player_id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0.00,0.00,0.00,0.00,0,0,1,Arsenal,Forward,Folarin Balogun
2,0,0,0,0,0,0,0,0,0,0,...,0.00,0.00,0.00,0.00,0,0,1,Arsenal,Defender,Cédric Alves Soares
3,0,0,0,0,0,0,0,0,0,0,...,0.00,0.00,0.00,0.00,0,0,1,Arsenal,Midfielder,Mohamed Elneny
4,0,0,0,0,0,0,0,0,0,0,...,0.00,0.00,0.00,0.00,0,0,1,Arsenal,Midfielder,Fábio Ferreira Vieira
5,4,0,0,0,0,0,0,0,0,0,...,0.00,0.00,0.00,0.02,1,0,1,Arsenal,Defender,Gabriel dos Santos Magalhães


In [112]:


a = df.query("in_dreamteam > 0")
a

a['player_name'].value_counts(ascending=False).plot(kind='bar')

,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,penalties_missed,yellow_cards,red_cards,...,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,total_points,in_dreamteam,gameweek,team,position,player_name
player_id,,,,,,,,,,,,,,,,,,,,,
19,90,1,0,0,1,0,0,0,0,0,...,0.19,0.17,0.36,1.18,10,1,1,Arsenal,Midfielder,Bukayo Saka
297,135,2,1,0,2,0,0,0,0,0,...,1.39,0.05,1.44,2.33,17,1,25,Liverpool,Forward,Cody Gakpo
60,90,1,1,0,2,0,0,0,0,0,...,1.40,0.04,1.44,2.58,9,1,26,Aston Villa,Forward,Ollie Watkins
43,90,2,0,0,2,0,0,0,0,0,...,0.55,0.05,0.60,2.58,15,1,26,Aston Villa,Midfielder,Douglas Luiz Soares de Paulo
34,90,1,1,0,2,0,0,0,0,0,...,1.05,0.66,1.71,2.58,12,1,26,Aston Villa,Midfielder,Leon Bailey
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,65,1,0,1,0,0,0,0,0,0,...,0.26,0.01,0.27,0.40,10,1,13,Bournemouth,Midfielder,Justin Kluivert
31,90,0,0,1,0,0,0,0,0,0,...,0.02,0.27,0.29,1.11,9,1,13,Arsenal,Defender,Oleksandr Zinchenko
664,90,0,2,0,2,0,0,0,0,0,...,0.15,0.57,0.72,1.68,11,1,12,West Ham,Midfielder,James Ward-Prowse
